# Scrap Serenity (@aleabitoreddit)

X(Twitter) 트윗 수집 → `analysis_Serenity.db`

- **post**: 일반 게시물
- **reply**: 답글
- **subscriber**: 구독자 전용 (`exclusivityInfo` 기반 감지)

## 1. Configuration

쿠키와 설정값을 여기서 수정하세요.

In [1]:
# ── Cookie ────────────────────────────────────────────────────────────────────
# X(Twitter) 쿠키 — 만료 시 브라우저에서 새 값 복사
COOKIES = {
    "auth_token": "0732cb0b867deebea6ad8ca223fffa92a82c51f8",
    "ct0": "b0d3fcc9aa34f7f41736d0856a23baa3758536c0be1be45de601877529e58ac9a7cf238ec42432f170eb88c641aa10db1f9095875d48587c84998df013ce9edda7a75d44891dfb8676fbb2367032d04d",
    "twid": "u%3D1446734216616501252"
}

# ── Target ────────────────────────────────────────────────────────────────────
TARGET_USER = "aleabitoreddit"

# ── Options ───────────────────────────────────────────────────────────────────
TWEET_COUNT = None   # None = 전체, 숫자 = 최대 N개
DEBUG = False        # True = raw tweet._data JSON 덤프

# ── Paths ─────────────────────────────────────────────────────────────────────
import os
_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
DB_DIR = os.path.join(_notebook_dir, 'Serenity', 'References')
DB_PATH = os.path.join(DB_DIR, 'analysis_Serenity.db')

os.makedirs(DB_DIR, exist_ok=True)
print(f"DB path: {DB_PATH}")

DB path: /Users/seongjin/Documents/Seongjin_Claude/Dumok-of-WallStreet/.claude/skills/Serenity/References/analysis_Serenity.db


## 2. Setup

In [2]:
import asyncio
import sqlite3
import json
import re
import tempfile
from datetime import datetime, timedelta, timezone
from twikit import Client

KST = timezone(timedelta(hours=9))
RATE_LIMIT_DELAY = 2
DUPLICATE_THRESHOLD = 10
MAX_RETRIES = 3


def to_kst(twitter_time_str):
    dt = datetime.strptime(twitter_time_str, "%a %b %d %H:%M:%S %z %Y")
    return dt.astimezone(KST).strftime("%Y-%m-%dT%H:%M:%S+09:00")


def extract_tickers(text):
    tickers = re.findall(r'\$([A-Za-z]+)', text)
    return list(dict.fromkeys([t.upper() for t in tickers]))


def get_full_content(tweet):
    note = (tweet._data
            .get('note_tweet', {})
            .get('note_tweet_results', {})
            .get('result', {}))
    if note and 'text' in note:
        return note['text']
    return tweet.full_text


def get_media_urls(tweet):
    urls = []
    if tweet.media:
        for m in tweet.media:
            if hasattr(m, 'media_url') and m.media_url:
                urls.append(m.media_url)
    return urls


def classify_tweet_type(tweet):
    if tweet._data.get('exclusivityInfo'):
        return 'subscriber'
    elif tweet.in_reply_to:
        return 'reply'
    else:
        return 'post'


def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS tweets (
            id TEXT PRIMARY KEY,
            user TEXT NOT NULL,
            type TEXT NOT NULL CHECK(type IN ('post', 'reply', 'subscriber')),
            created_at TEXT NOT NULL,
            content TEXT,
            tickers TEXT DEFAULT '[]',
            media TEXT DEFAULT '[]'
        )
    ''')
    conn.commit()
    return conn


def get_existing_ids(conn):
    cur = conn.execute('SELECT id FROM tweets')
    return set(row[0] for row in cur.fetchall())


def save_tweets(conn, tweets):
    saved = 0
    for tweet in tweets:
        content = get_full_content(tweet)
        tickers = json.dumps(extract_tickers(content), ensure_ascii=False)
        media = json.dumps(get_media_urls(tweet), ensure_ascii=False)
        tweet_type = classify_tweet_type(tweet)
        conn.execute(
            'INSERT OR IGNORE INTO tweets (id, user, type, created_at, content, tickers, media) VALUES (?,?,?,?,?,?,?)',
            (tweet.id, tweet.user.screen_name if tweet.user else TARGET_USER,
             tweet_type, to_kst(tweet.created_at), content, tickers, media))
        if conn.total_changes:
            saved += 1
    conn.commit()
    return saved


async def fetch_with_retry(coro_fn):
    for attempt in range(MAX_RETRIES):
        try:
            return await coro_fn()
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  ⚠ {e} — retry in {wait}s...")
            await asyncio.sleep(wait)


print("Setup complete.")

Setup complete.


## 3. Scrape

In [3]:
async def run_scrape():
    conn = init_db()
    existing_ids = get_existing_ids(conn)
    print(f"DB: {DB_PATH}")
    print(f"Existing: {len(existing_ids)}")

    # 쿠키를 임시 파일로 저장하여 twikit에 전달
    cookie_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
    json.dump(COOKIES, cookie_file)
    cookie_file.close()

    client = Client('en-US')
    try:
        client.load_cookies(cookie_file.name)
    except Exception as e:
        print(f"✗ Cookie expired — update COOKIES in cell 1.\n  {e}")
        conn.close()
        return
    finally:
        os.unlink(cookie_file.name)

    user = await client.get_user_by_screen_name(TARGET_USER)
    print(f"\n=== {user.name} (@{user.screen_name}) ===")
    print(f"Followers: {user.followers_count:,} | Tweets: {user.statuses_count:,}")

    total_saved = 0
    type_counts = {'post': 0, 'reply': 0, 'subscriber': 0}

    for tweet_type in ['Tweets', 'Replies']:
        print(f"\nFetching {tweet_type}...")
        consecutive_dups = 0
        total_fetched = 0

        try:
            tweets = await fetch_with_retry(
                lambda tt=tweet_type: user.get_tweets(tt, count=20))
        except Exception as e:
            print(f"  ⚠ Failed: {e}")
            continue

        while tweets:
            page_tweets = []
            for t in tweets:
                sn = t.user.screen_name if t.user else None
                if sn and sn.lower() != TARGET_USER.lower():
                    continue

                total_fetched += 1

                if DEBUG:
                    print(f"  [DEBUG] {t.id}: exclusivityInfo={t._data.get('exclusivityInfo')}, in_reply_to={t.in_reply_to}")

                if t.id in existing_ids:
                    consecutive_dups += 1
                    if consecutive_dups >= DUPLICATE_THRESHOLD:
                        print(f"  {DUPLICATE_THRESHOLD} consecutive dups — stopping")
                        break
                else:
                    consecutive_dups = 0
                    tt = classify_tweet_type(t)
                    type_counts[tt] += 1
                    page_tweets.append(t)
                    existing_ids.add(t.id)
                    if TWEET_COUNT and (total_saved + len(page_tweets)) >= TWEET_COUNT:
                        break

            if page_tweets:
                saved = save_tweets(conn, page_tweets)
                total_saved += saved
                print(f"  Fetched {total_fetched}, saved {total_saved} total")

            if consecutive_dups >= DUPLICATE_THRESHOLD:
                break
            if TWEET_COUNT and total_saved >= TWEET_COUNT:
                break

            await asyncio.sleep(RATE_LIMIT_DELAY)
            try:
                tweets = await fetch_with_retry(lambda: tweets.next())
                if not tweets:
                    break
            except Exception as e:
                print(f"  ⚠ Pagination stopped: {e}")
                break

        if TWEET_COUNT and total_saved >= TWEET_COUNT:
            break

    print(f"\n{'='*40}")
    print(f"Saved this run: {total_saved}")
    print(f"  post={type_counts['post']}, reply={type_counts['reply']}, subscriber={type_counts['subscriber']}")

    # DB 통계
    cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
    stats = cur.fetchall()
    total = sum(c for _, c in stats)
    print(f"\nTotal in DB: {total}")
    for t, c in stats:
        print(f"  {t}: {c}")
    conn.close()

await run_scrape()

DB: /Users/seongjin/Documents/Seongjin_Claude/Dumok-of-WallStreet/.claude/skills/Serenity/References/analysis_Serenity.db
Existing: 830

=== Serenity (@aleabitoreddit) ===
Followers: 125,475 | Tweets: 4,972

Fetching Tweets...
  Fetched 18, saved 18 total
  Fetched 38, saved 38 total
  Fetched 56, saved 56 total
  Fetched 76, saved 68 total
  10 consecutive dups — stopping

Fetching Replies...
  10 consecutive dups — stopping

Saved this run: 68
  post=59, reply=0, subscriber=9

Total in DB: 898
  post: 770
  reply: 1
  subscriber: 127


## 4. Explore DB

In [4]:
# ── 최신 트윗 확인 ─────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    'SELECT id, type, created_at, substr(content,1,80), tickers FROM tweets ORDER BY created_at DESC LIMIT 10')
for row in cur:
    print(f"[{row[1]:10s}] {row[2]}  {row[3]}")
    if row[4] != '[]':
        print(f"             tickers: {row[4]}")
conn.close()

[post      ] 2026-03-29T13:21:59+09:00  Maybe it’s time to start looking into the Doomsday ETF I made, as a hedge with $
             tickers: ["NVDA", "LCID"]
[post      ] 2026-03-29T12:15:16+09:00  Can't $SIVE just pull a $LITE and TAM expansion down the ELS stack?

I'm getting
             tickers: ["SIVE", "LITE", "AXTI", "AVGO", "META", "AMZN", "MSFT", "NVDA", "COHR", "POET", "FN"]
[post      ] 2026-03-29T02:26:42+09:00  Honestly... the most obvious ideas like long $LNG.

Or going long on $CVX.

Are 
             tickers: ["LNG", "CVX"]
[post      ] 2026-03-29T01:44:25+09:00  Iran has now just drone striked the major UAE Aluminum Plant. 

If your first th
             tickers: ["CNEX", "AA"]
[post      ] 2026-03-29T00:32:25+09:00  Faster compounds:

$AAOI - 10x revenue ramp from optical transcivers h2 2027
$NB
             tickers: ["AAOI", "NBIS", "ARM", "MRVL", "MSFT", "AVGO", "LITE", "TSEM", "VNP", "NEO", "AMZN", "CRCL", "RDDT", "GLD", "IBIT", "CVX", "INTC", "AMKR", "SOI", "RKL

In [5]:
# ── 타입별 통계 ───────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
for t, c in cur:
    print(f"  {t}: {c}")
total = conn.execute('SELECT COUNT(*) FROM tweets').fetchone()[0]
print(f"  total: {total}")
conn.close()

  post: 770
  reply: 1
  subscriber: 127
  total: 898


In [6]:
# ── 구독자 전용 트윗 확인 ─────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT created_at, substr(content,1,100), tickers FROM tweets WHERE type='subscriber' ORDER BY created_at DESC LIMIT 10")
for row in cur:
    print(f"{row[0]}  {row[1]}")
    if row[2] != '[]':
        print(f"  tickers: {row[2]}")
    print()
conn.close()

2026-03-28T22:03:00+09:00  SpaceX Supply Chain ahead of IPO list: 

There's way too many companies LOL, need to go through each
  tickers: ["ATRO", "DCO", "HEI", "ATI", "VELO", "SIF", "PH", "MOG", "LIN", "APD", "TDG", "GTLS", "FLTCF", "HXL", "TRYIY", "PKE", "STLD", "CRS", "MTRN", "GHM", "STM", "AVGO", "CMTL", "TRMB", "M", "MPWR", "TXN", "DIOD"]

2026-03-28T11:48:06+09:00  So some thoughts on Iran/Trade:

I messed up and underestimated Trump decisive strikes like Venezuel
  tickers: ["GOOGL"]

2026-03-27T12:03:35+09:00  So first long position:

Win Semi - $4B MC. I keep talking about this name over and over…

Especiall
  tickers: ["SIVE", "LITE", "TSM", "AAOI"]

2026-03-27T02:38:05+09:00  Yeah… think markets are pricing a ground invasion (no TACO) and potential oil shock, hence all the s
  tickers: ["CVX"]

2026-03-26T23:06:24+09:00  I went personally long on:

$MSFT $372 and $META $576 again.

Looks too discounted to me? 

Probably
  tickers: ["MSFT", "META"]

2026-03-26T14:35:20+09:00

In [7]:
# ── 특정 티커 검색 ────────────────────────────────────────────────────────
SEARCH_TICKER = "AXTI"  # ← 변경하여 검색

conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT type, created_at, substr(content,1,100) FROM tweets WHERE tickers LIKE ? ORDER BY created_at DESC",
    (f'%"{SEARCH_TICKER}"%',))
rows = cur.fetchall()
print(f"${SEARCH_TICKER} mentions: {len(rows)}")
for row in rows[:10]:
    print(f"  [{row[0]}] {row[1]}  {row[2]}")
conn.close()

$AXTI mentions: 134
  [post] 2026-03-29T12:15:16+09:00  Can't $SIVE just pull a $LITE and TAM expansion down the ELS stack?

I'm getting early stage $LITE f
  [post] 2026-03-29T00:32:25+09:00  Faster compounds:

$AAOI - 10x revenue ramp from optical transcivers h2 2027
$NBIS - 10x revenue ram
  [post] 2026-03-28T11:17:36+09:00  My portfolio has drawdowns from Macro as well.

YTD is now 527%.

After the index crashed -7%. 

- $
  [post] 2026-03-28T10:10:05+09:00  Great callout on $HIMX and now down -31.54% (accelerated by macro). 

This is a good lesson around c
  [post] 2026-03-27T20:32:44+09:00  This is why you need to have conviction before entering a trade. 

If you knew $SIVE positioning in 
  [post] 2026-03-27T11:13:44+09:00  This crash today is a good reason why it's good to stay invested.

-> $AAOI is still up 233%?
-> $IQ
  [post] 2026-03-27T01:12:55+09:00  Photonics are not having a fun time.

Laser Companies from:
$LITE, $SIVE, $COHR, $MTSI, $AAOI all do
  [post] 2026-03-26T1